### Paper Analysis Notebook - Results

This notebook is for plotting some of the figures in the paper and interpreting the results of the model simulation.

Prerequisites:
    
    -  to run this notebook you need to have first run "full_simulatyion.ipynb

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##### 1. User Config

In [ ]:
# Some constant parameters
total_capstock = 1.227*1e12 # from the Thai national capital stock tables
baseline_rating = 12.42 # this is what model predicts with no GDP losses (actual S&P rating is 13)

##### 2. Load and prepare data

In [ ]:
root = Path.cwd().parent.parent # find project root
# Load results
df = pd.read_csv(os.path.join(root, 'outputs', 'results', 'full_model_simulation.csv'))
# Add total cap damage column
df["CAP_dam"] = df[["PRI_dam", "PUB_dam"]].sum(axis=1)
# Add downgrade column
df["Downgrade"] = baseline_rating - df["predicted_rating"]

##### 3. Define common functions

In [ ]:
# Functions for extracting future scenarios
def get_sorted_series(df, scenario, epoch, stat, adaptation, col):
    d = df[
        (df["scenario"].eq(scenario)) &
        (df["adaptation"].eq(adaptation)) &
        ((df["epoch"].eq(epoch)) if epoch is not None else df["epoch"].isna()) &
        ((df["stat"].eq(stat)) if stat is not None else df["stat"].isna())
    ][[col]].copy()

    d = d.sort_values(col, ascending=False).reset_index(drop=True)
    return d[col].to_numpy()

### Table 1 - Baseline mean (AAL) and Q99 (RP100) impacts

In [ ]:
# Table reporting mean and Q99 impacts across metric distributions
baseline_losses = df[
    (df["scenario"].eq("baseline")) &
    (df["adaptation"].eq(False))
].copy()

baseline_impact_series = pd.DataFrame({
    "Capital stock loss (USD bn)": baseline_losses["CAP_dam"] / 1e9,
    "GDP loss (%)": baseline_losses["gdp_loss"] * -1,
    "Credit rating impact (notches)": baseline_losses["Downgrade"],
    "Probability of default increase (percentage points)": baseline_losses["predicted_pd"] - baseline_losses["original_pd"],
    "Cost of debt increase (basis points)": baseline_losses["spread_delta"],
})

baseline_mean_q99_table = pd.DataFrame({
    "Mean": baseline_impact_series.mean(axis=0),
    "Q99": baseline_impact_series.quantile(0.99, axis=0),
})

baseline_mean_q99_table.round(2)


#### Figure 4 - Full model chain outputs (baseline scenario)

In [ ]:
order_col = "CAP_dam" # what metric do we want to order losses by?
plot_col = 'navy'
figure_4 = df[
    (df['scenario'] == 'baseline') &
    (df['adaptation'] == False)
    ]
# Sort on order col
figure_4_sorted = figure_4.sort_values(by=order_col, ascending=False).reset_index(drop=True)
figure_4_sorted['order'] = np.arange(len(figure_4_sorted))
# Extract data to plot
year_index = figure_4_sorted['year_index'].to_numpy()
cap_series = figure_4_sorted['CAP_dam'].to_numpy()
gdp_series = figure_4_sorted['gdp_loss'].to_numpy()
rat_series = figure_4_sorted['Downgrade'].to_numpy()
x = np.arange(1, len(cap_series)+1) / (len(cap_series)+1) # x values for plot

# Plot figure
fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(7,9),
    sharex=True
)
# axes[0].scatter(x, (cap_series/total_capstock)*100, s=0.1, color=plot_col)
axes[0].scatter(x, cap_series/1e9, s=0.1, color=plot_col)
axes[0].set_ylabel("Capital Loss (USD $B)")
axes[0].set_xscale("log")
axes[1].scatter(x, gdp_series*-1, s=0.1, color=plot_col)
axes[1].set_ylabel("GDP Loss (%)")
axes[2].scatter(x, rat_series, s=0.1, color=plot_col)
axes[2].set_ylabel("Predicted Downgrade (Notches)")
axes[2].set_xlabel("Capital Loss Exceedance Probability") # All ordered on cap loss so just use this X label once at bottom

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.001, 1)

plt.subplots_adjust(hspace=0.05) # adjust spacing

plt.show()

### Table 2 - Future AAL and RP100 changes

In [ ]:
# Table 2: future AAL and RP100 changes
future_table_loss_type = "GDP"  # options: "Capital", "GDP"
future_metric_loss_settings = {
    "Capital": {"col": "CAP_dam", "multiplier": 1e-9},
    "GDP": {"col": "gdp_loss", "multiplier": -1},
}
future_metric_defs = {
    "AAL": None,
    "RP100": 100,
}
future_scenarios = ["ssp126", "ssp370", "ssp585"]
future_table_epochs = [2030, 2050, 2070]
future_stats = ["mean", "q05", "q95"]
future_adaptation_labels = {
    False: "Unadapted",
    True: "Adapted",
}


def distribution_metric_value(losses, return_period):
    losses = pd.Series(losses).dropna()
    if return_period is None:
        return losses.mean()
    return losses.quantile(1 - 1 / return_period)


def format_change_number(value):
    if np.isclose(value, 0, atol=0.05):
        return "0"
    return f"{value:.1f}"


def format_metric_change(row):
    low = min(row["q05 change from baseline (%)"], row["q95 change from baseline (%)"])
    high = max(row["q05 change from baseline (%)"], row["q95 change from baseline (%)"])
    mean = row["mean change from baseline (%)"]
    return f'{format_change_number(mean)} [{format_change_number(low)} - {format_change_number(high)}]'


settings = future_metric_loss_settings[future_table_loss_type]
col = settings["col"]
multiplier = settings["multiplier"]
baseline_losses = df[
    (df["scenario"].eq("baseline")) &
    (df["adaptation"].eq(False))
][col] * multiplier

baseline_metrics = {
    metric: distribution_metric_value(baseline_losses, return_period)
    for metric, return_period in future_metric_defs.items()
}

future_metric_rows = []
for scenario in future_scenarios:
    for epoch in future_table_epochs:
        for adaptation, adaptation_label in future_adaptation_labels.items():
            for metric, return_period in future_metric_defs.items():
                row = {
                    "Loss type": future_table_loss_type,
                    "Scenario": scenario.upper(),
                    "Epoch": epoch,
                    "Protection": adaptation_label,
                    "Metric": metric,
                    "Baseline value": baseline_metrics[metric],
                }

                for stat in future_stats:
                    future_losses = df[
                        (df["scenario"].eq(scenario)) &
                        (df["epoch"].eq(epoch)) &
                        (df["stat"].eq(stat)) &
                        (df["adaptation"].eq(adaptation))
                    ][col] * multiplier
                    value = distribution_metric_value(future_losses, return_period)
                    row[f"{stat} value"] = value
                    row[f"{stat} change from baseline (%)"] = ((value / baseline_metrics[metric]) - 1) * 100

                future_metric_rows.append(row)

future_metric_changes = pd.DataFrame(future_metric_rows)
future_metric_changes["Change from baseline (%)"] = future_metric_changes.apply(format_metric_change, axis=1)

future_metric_change_table = (
    future_metric_changes
    .pivot(
        index=["Scenario", "Epoch"],
        columns=["Protection", "Metric"],
        values="Change from baseline (%)",
    )
    .loc[
        pd.MultiIndex.from_product(
            [[scenario.upper() for scenario in future_scenarios], future_table_epochs],
            names=["Scenario", "Epoch"],
        ),
        pd.MultiIndex.from_product(
            [["Unadapted", "Adapted"], ["AAL", "RP100"]],
            names=["", ""],
        ),
    ]
)
future_metric_change_table.columns.names = [None, None]

future_metric_change_table

### Figure 5 - Downgrade probabilities over time with 90% window

In [ ]:
### Run some calcs to calculate downgrade
# Figure information
stats = ["mean", "q05", "q95"] # use mean point estimate with q05-q95 window
epochs = [2030, 2040, 2050, 2060, 2070] # epochs to plot
scenarios = ["ssp126", "ssp370", "ssp585"] # scenarios to plot
adaptation_vals = [True, False] # adaptation scenarios to plot
mask_future = (
    (df["stat"].isin(stats)) &
    (df["epoch"].isin(epochs)) &
    (df["scenario"].isin(scenarios)) &
    (df["adaptation"].isin(adaptation_vals))
)
mask_baseline = df["scenario"].eq("baseline") & df["adaptation"].eq(False)
downgrades_window = df.loc[mask_future | mask_baseline].copy()

downgrades_window = downgrades_window[["scenario", "epoch", "stat", "Downgrade", "year_index", "adaptation"]]

def downgrade_stats(group):
    x = group["Downgrade"]
    return pd.Series({
        "P_1plus": (x >= 1).mean() * 100, # probability of at least a 1 notch downgrade
        "P_3plus": (x >= 3).mean() * 100, # probability of a 3 notch downgrade or worse (sub-investment grade downgtade)
    })

summary_window = (
    downgrades_window
    .groupby(["scenario", "epoch", "adaptation", "stat"], dropna=False)
    .apply(downgrade_stats)
    .reset_index()
)

dfp_window = summary_window.copy()

plot_vars = ["P_1plus", "P_3plus"] # which variables are we plotting

# split baseline vs futures
base = dfp_window[dfp_window["scenario"].eq("baseline")].copy()
fut = dfp_window[dfp_window["scenario"].ne("baseline")].copy()
fut["epoch"] = fut["epoch"].astype(int)

fig, axes = plt.subplots(2, 3, figsize=(12,7), sharex=True)

for row, var in enumerate(plot_vars):
    for col, sc in enumerate(scenarios):

        ax = axes[row, col]
        d = fut[fut["scenario"].eq(sc)].sort_values("epoch")

        for ad in [False, True]:
            dd = (
                d[d["adaptation"].eq(ad)]
                .pivot(index="epoch", columns="stat", values=var)
                .reindex(epochs)
            )

            if ad == True:
                color = "blue"
                label = "Adapted"
            if ad == False:
                color = "#D55E00"
                label = "Unadapted"

            lower = np.minimum(dd["q05"], dd["q95"])
            upper = np.maximum(dd["q05"], dd["q95"])
            ax.fill_between(
                dd.index.to_numpy(),
                lower.to_numpy(),
                upper.to_numpy(),
                color=color,
                alpha=0.15,
                linewidth=0,
            )
            ax.plot(
                dd.index,
                dd["mean"],
                marker="o",
                color=color,
                label=label,
            )

        # baseline line
        b = base[base["adaptation"].eq(False)][var].iloc[0]
        ax.axhline(b, linestyle="--", linewidth=1, color="k", label="Baseline")

        if row == 0:
            ax.set_title(sc, size=14)

        if col == 0 and row == 0:
            ax.set_ylabel("P(downgrade) (%)" if "P_" in var else var, size=14)

        if col == 0 and row == 1:
            ax.set_ylabel("P(sub-IG) (%)" if "P_" in var else var, size=14)

        # Hide Y labels
        if col != 0:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

# single legend
axes[0,0].legend(loc="upper left", bbox_to_anchor=(0,1))

# Set axes sizes from the full q05-mean-q95 range so the windows are not clipped
for row, var in enumerate(plot_vars):
    vals = pd.concat([fut[var], base[var]]).dropna()
    ymin = vals.min()
    ymax = vals.max()
    margin = (ymax - ymin) * 0.1 if ymax > ymin else max(ymax * 0.1, 0.1)
    for ax in axes[row]:
        ax.set_ylim(max(0, ymin - margin), ymax + margin)

plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic table of the values plotted in Figure 5
figure5_metric_labels = {
    "P_1plus": "P(downgrade >= 1 notch) (%)",
    "P_3plus": "P(sub-IG downgrade >= 3 notches) (%)",
}
figure5_adaptation_labels = {False: "Unadapted", True: "Adapted"}

figure5_future_table = (
    fut
    .melt(
        id_vars=["scenario", "epoch", "adaptation", "stat"],
        value_vars=plot_vars,
        var_name="metric",
        value_name="probability_pct",
    )
    .assign(metric=lambda x: x["metric"].map(figure5_metric_labels))
    .pivot_table(
        index=["scenario", "epoch", "adaptation", "metric"],
        columns="stat",
        values="probability_pct",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["metric", "scenario", "adaptation", "epoch"])
)

figure5_future_table["window"] = figure5_future_table.apply(
    lambda r: f"{min(r['q05'], r['q95']):.2f}-{max(r['q05'], r['q95']):.2f}",
    axis=1,
)
figure5_future_table["adaptation"] = figure5_future_table["adaptation"].map(figure5_adaptation_labels)
figure5_future_table.columns.name = None
figure5_future_table = figure5_future_table[
    ["scenario", "epoch", "adaptation", "metric", "mean", "q05", "q95", "window"]
]

figure5_baseline_table = pd.DataFrame({
    "metric": [figure5_metric_labels[var] for var in plot_vars],
    "probability_pct": [base[base["adaptation"].eq(False)][var].iloc[0] for var in plot_vars],
})

print("Figure 5 future values: mean line and q05-q95 window")
display(figure5_future_table.round({"mean": 2, "q05": 2, "q95": 2}))

print("Figure 5 baseline dashed reference lines")
display(figure5_baseline_table.round({"probability_pct": 2}))

### Results: Adaptation Benefits

In [ ]:
# Adaptation AAL benefits across all future scenario, epoch, and quantile/stat combinations
# Positive values mean adaptation reduces losses: unadapted AAL - adapted AAL.
adaptation_benefit_loss_type = "Capital"  # options: "GDP", "Capital"

adaptation_benefit_settings = {
    "GDP": {"col": "gdp_loss", "multiplier": -1, "unit": "GDP loss (%)"},
    "Capital": {"col": "CAP_dam", "multiplier": 1e-9, "unit": "Capital loss (USD bn)"},
}

adaptation_benefit_config = adaptation_benefit_settings[adaptation_benefit_loss_type]
adaptation_benefit_future = df.loc[df["scenario"].ne("baseline")].copy()
adaptation_benefit_future["_epoch_int"] = pd.to_numeric(adaptation_benefit_future["epoch"], errors="coerce").astype("Int64")

adaptation_benefit_stat_order = {"q05": 0, "q50": 1, "mean": 2, "q95": 3}
adaptation_benefit_combinations = (
    adaptation_benefit_future[["scenario", "_epoch_int", "stat"]]
    .dropna()
    .drop_duplicates()
    .assign(
        stat_sort=lambda frame: frame["stat"].map(
            lambda stat: adaptation_benefit_stat_order.get(str(stat), len(adaptation_benefit_stat_order))
        )
    )
    .sort_values(["scenario", "_epoch_int", "stat_sort", "stat"])
)

def get_adaptation_benefit_aal(scenario, epoch, stat, adaptation):
    selection = adaptation_benefit_future[
        adaptation_benefit_future["scenario"].eq(scenario)
        & adaptation_benefit_future["_epoch_int"].eq(epoch)
        & adaptation_benefit_future["stat"].eq(stat)
        & adaptation_benefit_future["adaptation"].eq(adaptation)
    ]
    if selection.empty:
        return np.nan

    return selection[adaptation_benefit_config["col"]].dropna().mean() * adaptation_benefit_config["multiplier"]

adaptation_benefit_rows = []
for _, combination in adaptation_benefit_combinations.iterrows():
    scenario = combination["scenario"]
    epoch = int(combination["_epoch_int"])
    stat = combination["stat"]
    unadapted_aal = get_adaptation_benefit_aal(scenario, epoch, stat, adaptation=False)
    adapted_aal = get_adaptation_benefit_aal(scenario, epoch, stat, adaptation=True)
    benefit = unadapted_aal - adapted_aal
    benefit_pct = np.nan if np.isclose(unadapted_aal, 0, equal_nan=True) else benefit / unadapted_aal * 100

    adaptation_benefit_rows.append({
        "Loss type": adaptation_benefit_loss_type,
        "Scenario": scenario.upper(),
        "Epoch": epoch,
        "Quantile/stat": stat,
        f"Unadapted AAL ({adaptation_benefit_config['unit']})": unadapted_aal,
        f"Adapted AAL ({adaptation_benefit_config['unit']})": adapted_aal,
        f"Adaptation benefit ({adaptation_benefit_config['unit']})": benefit,
        "Adaptation benefit (% reduction)": benefit_pct,
    })

adaptation_benefit_table = pd.DataFrame(adaptation_benefit_rows)
adaptation_benefit_value_col = f"Adaptation benefit ({adaptation_benefit_config['unit']})"

adaptation_benefit_range = (
    adaptation_benefit_table
    .groupby("Quantile/stat", dropna=False)
    .agg(
        combinations=("Scenario", "count"),
        min_benefit=(adaptation_benefit_value_col, "min"),
        max_benefit=(adaptation_benefit_value_col, "max"),
        min_pct_reduction=("Adaptation benefit (% reduction)", "min"),
        max_pct_reduction=("Adaptation benefit (% reduction)", "max"),
    )
    .reset_index()
)

print(f"AAL adaptation benefit range across all future scenario/epoch combinations ({adaptation_benefit_loss_type})")
display(adaptation_benefit_range.round({
    "min_benefit": 2,
    "max_benefit": 2,
    "min_pct_reduction": 1,
    "max_pct_reduction": 1,
}))
